# Smart City V2 — 02 Route Runner

**Node 1 luôn là START.** Click Node 4 hoặc 2 để chọn hướng đặt xe, rồi click các neighbor tiếp theo. Chuyển động hoàn toàn theo config thời gian; traffic light chỉ dùng OpenCV HSV + contour area + circularity trên nửa trên ảnh.

> Mặc định DISARMED. Hãy thử kê bánh trước; kiểm tra route preview và các phase quẹo trước khi RUN trên grid.

In [ ]:
from pathlib import Path
import copy, html, sys, threading, time
import cv2, numpy as np, ipywidgets as widgets
from IPython.display import display

cwd = Path.cwd().resolve()
WORKDIR = cwd if (cwd / 'smart_city_v2_core.py').exists() else cwd / 'smart city v2'
if not (WORKDIR / 'smart_city_v2_core.py').exists():
    raise FileNotFoundError('Open this notebook from yolo_lane_following or smart city v2')
if str(WORKDIR) not in sys.path: sys.path.insert(0, str(WORKDIR))
from smart_city_v2_core import (FIXED_THROTTLE, POSITIONS, START_NODE, MotionDriver,
    RouteExecutor, TrafficLightDetector, apply_command_tuning, are_neighbors,
    classify_turn, compile_route, load_config, save_config)

CONFIG_PATH = WORKDIR / 'track_config.json'
config = load_config(CONFIG_PATH, require_measurements=True)
print(f'Loaded {CONFIG_PATH}')
print(f'Throttle {FIXED_THROTTLE:.2f} — LOCKED')

## Xe + camera traffic light
Camera chỉ xét `frame[:height//2]`. RED đã confirm sẽ khóa pause; sau đó NONE hoặc RED vẫn đứng yên, chỉ GREEN đã confirm mới resume.

In [ ]:
from jetracer.nvidia_racecar import NvidiaRacecar
from jetcam.csi_camera import CSICamera

# Release stale camera observer when re-running this cell.
try:
    camera.running = False; camera.unobserve_all()
except Exception:
    pass
car = NvidiaRacecar()
car.steering_gain = float(config['motion'].get('steering_gain', -0.65))
car.steering_offset = float(config['motion'].get('steering_offset', 0.0))
car.throttle_gain = float(config['motion'].get('throttle_gain', 0.8))
car.throttle = 0.0; car.steering = float(config['motion']['steering_center'])
arm = widgets.Checkbox(value=False, description='ARM MOTOR')
driver = MotionDriver(car, armed=lambda: bool(arm.value))
driver.stop(center=True)

camera = CSICamera(width=224, height=224, capture_fps=30)
detector = TrafficLightDetector(config.get('traffic_light', {}))
camera_view = widgets.Image(format='jpeg', width=224, height=224)
light_label = widgets.HTML('<b>LIGHT: NONE</b>')
camera_status = widgets.HTML('<b style="color:#b60">Waiting for camera frames</b>')
light_lock = threading.Lock()
light_data = {'state': 'NONE', 'frames': 0, 'estimate': None}

def camera_update(change):
    frame = change['new']
    estimate = detector.update(frame)
    with light_lock:
        light_data.update(state=estimate.state, frames=light_data['frames']+1, estimate=estimate)
    debug = frame.copy()
    cv2.line(debug, (0, estimate.roi_height), (debug.shape[1]-1, estimate.roi_height), (255,255,255), 1)
    color = (0,0,255) if estimate.state == 'RED' else ((0,255,0) if estimate.state == 'GREEN' else (170,170,170))
    cv2.putText(debug, estimate.state, (6,18), cv2.FONT_HERSHEY_SIMPLEX, .55, color, 2)
    ok, encoded = cv2.imencode('.jpg', debug)
    if ok: camera_view.value = encoded.tobytes()
    light_label.value = (f'<b>LIGHT: <span style="color:{"red" if estimate.state=="RED" else "green" if estimate.state=="GREEN" else "#777"}">{estimate.state}</span></b>'
                         f' | red area={estimate.red_area:.0f}, circ={estimate.red_circularity:.2f}'
                         f' | green area={estimate.green_area:.0f}, circ={estimate.green_circularity:.2f}')
    camera_status.value = '<span style="color:#080">Camera running — detection ROI is upper half only</span>'

def current_light():
    with light_lock: return str(light_data['state'])

def on_arm(change):
    if not change['new']: driver.stop(center=True)
arm.observe(on_arm, names='value')
camera.observe(camera_update, names='value'); camera.running = True
display(widgets.VBox([widgets.HBox([camera_view, widgets.VBox([arm, camera_status, light_label])])]))

## Chọn route — Node 1 đã là bước ①
`CLEAR` luôn trở về `[1]`. Click đầu chỉ có thể là 4 hoặc 2. U-turn tức thời và node đã dùng bị chặn.

In [ ]:
route = [START_NODE]
route_buttons = {node: widgets.Button(description=str(node), layout=widgets.Layout(width='86px', height='54px')) for node in POSITIONS}
undo_button = widgets.Button(description='UNDO', icon='undo')
clear_button = widgets.Button(description='CLEAR', button_style='warning', icon='trash')
route_text = widgets.HTML()
place_text = widgets.HTML()
route_error = widgets.HTML()
command_box = widgets.VBox()
compiled_commands = []
command_widgets = []
running = {'value': False}
circled = ['⓪','①','②','③','④','⑤','⑥','⑦','⑧','⑨','⑩','⑪','⑫','⑬','⑭','⑮','⑯','⑰','⑱','⑲','⑳']

def route_mark(index): return circled[index] if index < len(circled) else f'[{index}]'

def bind_float(widget, command, field):
    def changed(change): setattr(command, field, float(change['new']))
    widget.observe(changed, names='value')

def rebuild_command_list():
    global compiled_commands, command_widgets
    command_widgets = []
    if len(route) < 2:
        compiled_commands = []
        command_box.children = (widgets.HTML('<i>Click Node 4 or Node 2 to establish FACE direction.</i>'),)
        return
    try:
        compiled_commands = compile_route(route, config)
    except Exception as exc:
        route_error.value = f'<b style="color:#b00">{html.escape(str(exc))}</b>'; return
    rows = []
    for index, command in enumerate(compiled_commands, 1):
        if command.kind == 'STRAIGHT':
            duration = widgets.BoundedFloatText(value=command.duration, min=0.01, max=20.0, step=0.01, description='seconds', layout=widgets.Layout(width='180px'))
            bind_float(duration, command, 'duration')
            title = widgets.HTML(f'<b>{index:02d} STRAIGHT &nbsp; {command.start} → {command.end}</b>')
            row = widgets.HBox([title, duration], layout=widgets.Layout(border='1px solid #ddd', padding='8px', justify_content='space-between'))
            command_widgets.append({'command': command, 'duration': duration})
        elif command.kind == 'TURN':
            pre = widgets.BoundedFloatText(value=command.pre_steer_time, min=0.0, max=2.0, step=0.01, description='1 PRE (s)', layout=widgets.Layout(width='175px'))
            steer = widgets.BoundedFloatText(value=command.steering, min=-1.0, max=1.0, step=0.01, description='STEERING', layout=widgets.Layout(width='175px'))
            arc = widgets.BoundedFloatText(value=command.duration, min=0.01, max=5.0, step=0.01, description='2 ARC (s)', layout=widgets.Layout(width='175px'))
            center = widgets.BoundedFloatText(value=command.center_settle_time, min=0.0, max=2.0, step=0.01, description='3 CENTER (s)', layout=widgets.Layout(width='185px'))
            bind_float(pre, command, 'pre_steer_time'); bind_float(steer, command, 'steering')
            bind_float(arc, command, 'duration'); bind_float(center, command, 'center_settle_time')
            title = widgets.HTML(f'<b>{index:02d} {command.turn} at Node {command.node}</b><br><small>Phase 2 ARC is the editable throttle-on time; throttle remains {FIXED_THROTTLE:.2f}.</small>')
            row = widgets.VBox([title, widgets.HBox([pre, steer, arc, center])], layout=widgets.Layout(border='2px solid #e89b2d', padding='9px'))
            command_widgets.append({'command': command, 'pre': pre, 'steer': steer, 'arc': arc, 'center': center})
        else:
            row = widgets.HTML(f'<div style="border:1px solid #b00;padding:8px"><b>{index:02d} STOP</b></div>')
        rows.append(row)
    command_box.children = tuple(rows)

def redraw_route(rebuild_commands=True):
    for node, button in route_buttons.items():
        if node in route:
            idx = route.index(node) + 1; button.description = route_mark(idx); button.button_style = 'success'
        else:
            button.description = str(node); button.button_style = ''
        legal = are_neighbors(route[-1], node) and node not in route
        if len(route) >= 2 and node == route[-2]: legal = False
        button.disabled = running['value'] or not legal
    route_buttons[START_NODE].disabled = True
    undo_button.disabled = running['value'] or len(route) == 1
    clear_button.disabled = running['value'] or len(route) == 1
    route_text.value = '<h3>ROUTE</h3><b>' + ' → '.join(map(str, route)) + '</b>'
    place_text.value = ('<h3>PLACE CAR</h3><b>START: NODE 1</b><br>' + (f'<b>FACE: NODE {route[1]}</b><br>1 → {route[1]}' if len(route)>1 else 'Click 4 or 2 for FACE direction'))
    route_error.value = ''
    if rebuild_commands: rebuild_command_list()

def click_node(node):
    def handler(_):
        if running['value'] or node in route or not are_neighbors(route[-1], node): return
        if len(route) >= 2:
            try: classify_turn(route[-2], route[-1], node)
            except Exception as exc:
                route_error.value = f'<b style="color:#b00">{html.escape(str(exc))}</b>'; return
        route.append(node); redraw_route()
    return handler
for node, button in route_buttons.items(): button.on_click(click_node(node))
def undo(_):
    if len(route) > 1 and not running['value']: route.pop(); redraw_route()
def clear(_):
    if not running['value']: route[:] = [START_NODE]; redraw_route()
undo_button.on_click(undo); clear_button.on_click(clear)
def hline(): return widgets.HTML('<div style="text-align:center;font-size:22px">─────</div>')
def vline(): return widgets.HTML('<div style="text-align:center;font-size:28px;line-height:34px">│</div>')
def blank(): return widgets.HTML('')
grid_items = [
    route_buttons[7],hline(),route_buttons[4],hline(),route_buttons[1],
    vline(),blank(),vline(),blank(),vline(),
    route_buttons[8],hline(),route_buttons[5],hline(),route_buttons[2],
    vline(),blank(),vline(),blank(),vline(),
    route_buttons[9],hline(),route_buttons[6],hline(),route_buttons[3],
]
grid = widgets.GridBox(grid_items, layout=widgets.Layout(grid_template_columns='86px 62px 86px 62px 86px', grid_template_rows='54px 34px 54px 34px 54px', grid_gap='0px'))
redraw_route()
display(widgets.HBox([widgets.VBox([grid, widgets.HBox([undo_button,clear_button]), route_error]), widgets.VBox([route_text,place_text])]))
display(widgets.HTML('<h3>COMPILED COMMANDS — editable before RUN</h3>'), command_box)

## Quan sát, tune rồi RUN
Sửa trực tiếp `2 ARC (s)` để thay số giây lên ga trong cú quẹo. `SAVE ROUTE TUNING` lưu theo đúng bộ ba node; `RUN ROUTE` luôn dùng các giá trị đang hiện, kể cả chưa save.

In [ ]:
save_tuning_button = widgets.Button(description='SAVE ROUTE TUNING', button_style='info', icon='save')
run_button = widgets.Button(description='▶ RUN ROUTE', button_style='success', icon='play')
emergency_button = widgets.Button(description='■ EMERGENCY STOP', button_style='danger', icon='stop')
run_status = widgets.HTML('<b>READY / DISARMED</b>')
progress = widgets.FloatProgress(value=0, min=0, max=1, description='Command')
executor = None; run_thread = None

def save_tuning(_):
    try:
        apply_command_tuning(config, compiled_commands)
        save_config(config, CONFIG_PATH, require_measurements=True)
        run_status.value = f'<b style="color:#080">Saved route-specific tuning to {CONFIG_PATH.name}</b>'
    except Exception as exc:
        run_status.value = f'<b style="color:#b00">Save failed: {html.escape(str(exc))}</b>'

def executor_update(info):
    state = info.get('state','')
    if state in ('FINISH','STOPPED'):
        running['value'] = False; redraw_route(rebuild_commands=False); run_button.disabled = False; save_tuning_button.disabled = False
        if state == 'FINISH':
            run_status.value = '<h3 style="color:#080">FINISH — stopped at final node</h3>'
        else:
            run_status.value = f'<h3 style="color:#b00">STOPPED — {html.escape(info.get("error",""))}</h3>'
        return
    duration = max(1e-9, float(info.get('command',{}).get('duration',1)))
    progress.value = min(1.0, float(info.get('completed',0))/duration)
    phase_names = {'TURN_PRE_STEER':'PHASE 1/3 PRE-STEER (throttle=0)', 'RUN_COMMAND':'DRIVE / PHASE 2 ARC', 'PAUSED_RED':'PAUSED RED — waiting confirmed GREEN', 'TURN_CENTER':'PHASE 3/3 CENTER (throttle=0, steering=0)'}
    color = '#b00' if state == 'PAUSED_RED' else '#075'
    run_status.value = (f'<b style="color:{color}">{phase_names.get(state,state)}</b> | command {info.get("index")}'
                        f' | completed={float(info.get("completed",0)):.3f}s | remaining={float(info.get("remaining",0)):.3f}s'
                        f' | light={info.get("light","NONE")}')

def run_route(_):
    global executor, run_thread
    if len(route) < 2:
        run_status.value = '<b style="color:#b00">Choose at least Node 4 or 2 after START.</b>'; return
    if not arm.value:
        run_status.value = '<b style="color:#b00">ARM MOTOR is off.</b>'; return
    with light_lock: frames = int(light_data['frames'])
    if not camera.running or frames < int(config.get('traffic_light',{}).get('confirm_frames',3)):
        run_status.value = '<b style="color:#b00">Camera/traffic-light detector is not ready.</b>'; return
    commands_to_run = copy.deepcopy(compiled_commands)  # exact values currently visible, saved or not
    running['value'] = True; redraw_route(rebuild_commands=False); run_button.disabled = True; save_tuning_button.disabled = True
    executor = RouteExecutor(driver, current_light, executor_update)
    run_thread = threading.Thread(target=executor.run, args=(commands_to_run,), daemon=True)
    run_thread.start()

def emergency_stop(_):
    if executor is not None: executor.emergency_stop()
    driver.stop(center=True); arm.value = False; running['value'] = False; redraw_route(rebuild_commands=False)
    run_status.value = '<h3 style="color:#b00">EMERGENCY STOP / DISARMED</h3>'

save_tuning_button.on_click(save_tuning); run_button.on_click(run_route); emergency_button.on_click(emergency_stop)
display(widgets.VBox([widgets.HBox([save_tuning_button,run_button,emergency_button]), progress, run_status]))

## Dừng an toàn — luôn chạy cell này trước khi đóng notebook

In [ ]:
try:
    if executor is not None: executor.emergency_stop()
except Exception:
    pass
driver.stop(center=True); arm.value = False
camera.running = False; camera.unobserve_all()
print('Stopped safely: throttle=0, steering=center, camera stopped, DISARMED.')